# NLP_02 — Embeddings et adaptation d’un transformer

*Conception pédagogique : Mathieu Weill. Usage personnel d’apprentissage ; droits et crédits tiers conservés.*

**Compétition de travail : [tweets](https://www.kaggle.com/competitions/nlp-getting-started)**

**Question : Quand un modèle préentraîné apporte-t-il quelque chose ?**

7 heures pédagogiques, hors pauses. Les exemples sont des jalons techniques courts ; l’essentiel du temps est consacré aux exercices, revues et expériences. **Version étudiant.**

**[Repère de compréhension : Comprendre embeddings, attention et adaptation](../../../course/chapitres/NLP_02/README.md)**


## Préparation et limites

Lire README.md, compétitions et règles. Les données officielles sont téléchargées personnellement et placées dans data/competitions/<slug>/. Aucun remplacement silencieux par des données simulées. Poids préentraînés à préparer avant NLP_02/CV_01/CV_02. Les limites de calcul et d’échanges dépendent du concours et du compte.

## Notions du jour

Un embedding est une représentation vectorielle apprise. La similarité dépend de cette représentation : elle ne constitue pas une vérité universelle sur le sens. Un transformer combine embeddings, positions, attention et transformations ; l’attention contextualise les tokens.

**Préentraînement** : apprentissage initial sur un grand corpus. **Extracteur gelé** : réutiliser les représentations sans modifier leurs poids. **Fine-tuning** : mettre à jour certains ou tous les poids sur la tâche Disaster Tweets. Une nouvelle tête de classification est initialisée aléatoirement.

Nous utilisons BERT-tiny anglophone pour limiter les ressources, pas comme recommandation de meilleur modèle. Le pooling moyen est masqué pour exclure le padding. Longueur, padding et batch influencent coût et mémoire. Un transfert préentraîné peut perdre face à TF-IDF, surtout sur un corpus jouet.

Préparation requise : `python -m mllab.prepare_assets --text` (Internet, avant la séance) et requirements-deep. En séance, chargement local par défaut. MLLAB_PRETRAINED=0 permet seulement de sauter les poids externes en secours ; ce saut ne valide pas le TP de transfert.

Travail principal sur les vrais tweets du concours, jamais sur les avis artificiels des annexes.

In [ ]:
from pathlib import Path
import sys,os,json,time
import numpy as np,pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
HERE=Path.cwd()
ROOT=next((p if (p/'mllab/competition.py').exists() else p/'student') for p in [HERE,*HERE.parents] if (p/'mllab/competition.py').exists() or (p/'student/mllab/competition.py').exists())
sys.path.insert(0,str(ROOT))
from mllab.competition import SPECS,read_competition,protocol,run_competition,binary_scores,get_folder
KEY='tweets';SPEC=SPECS[KEY]
QUICK=os.environ.get('MLLAB_QUICK','0')=='1'
LIMIT=int(os.environ.get('MLLAB_LIMIT','1000' if QUICK else '15000'))
IMAGE_LIMIT=int(os.environ.get('MLLAB_IMAGE_LIMIT','24' if QUICK else '240'))
plt.rcParams.update({'figure.figsize':(10,4),'axes.spines.top':False,'axes.spines.right':False})
print('Compétition :',SPEC['slug'],'métrique :',SPEC['metric'],'sens :',SPEC['direction'])
if KEY!='catsdogs':a,test,sample,protocol_record=protocol(KEY)
else:
 from mllab.vision_competition import image_protocol
 image_protocol(limit=IMAGE_LIMIT)
print('Données officielles trouvées. Profil rapide :',QUICK,'; aucun score Kaggle inventé.')

## Attention miniature pour comprendre le calcul

In [ ]:
import torch
Q=torch.tensor([[1.,0.],[0.,1.],[1.,1.]])
attention=torch.softmax(Q@Q.T/np.sqrt(2),dim=-1)
display(pd.DataFrame(attention.numpy()));assert torch.allclose(attention.sum(1),torch.ones(3))
print('Dans le vrai transformer, Q/K/V sont des projections apprises. Ici elles sont fixées pour l’intuition.')

## Encodeur gelé sur Disaster Tweets

In [ ]:
from mllab.text_competition import run_text
text_limit=min(LIMIT,300 if QUICK else 1500)
out,scores,history=run_text(limit=text_limit,mode='frozen',allow_download=False)
display(scores)
print('BERT-tiny anglais, poids préchargés ; pooling masqué puis classifieur linéaire.')

## Adapter et comparer sous le même budget

In [ ]:
out2,scores2,history=run_text(limit=text_limit,mode='finetune',epochs=1 if QUICK else 3,allow_download=False)
base,base_scores,_=run_competition(KEY,limit=text_limit)
display(pd.DataFrame([{'méthode':'TF-IDF',**base_scores},{'méthode':'encodeur gelé',**scores},{'méthode':'fine-tuning',**scores2}]))
print('Même protocole et mêmes observations. Un run court peut perdre ; analyser coût et erreurs avant de conclure.')

## Atelier autonome : produire des preuves

1. Expliquer quelles matrices sont apprises dans l’attention, contrairement à la démonstration Q/K/V fixés.
2. Comparer TF-IDF, extracteur gelé et fine-tuning sur le même vrai corpus si disponible ; sinon conclure uniquement sur le jouet.
3. Préparer un cache d’embeddings indexé par texte, révision du modèle, pooling et longueur ; vérifier une invalidation lors d’un changement.

Pour chaque essai : hypothèse avant calcul, code/configuration, empreinte du split, métrique officielle, durée, budget, conclusion, prochaine expérience. Chaque membre doit pouvoir expliquer son apport.

In [ ]:
HYPOTHESE = 'À écrire avant le prochain calcul'
CONCLUSION = 'À compléter avec une preuve et une limite'
# Ajouter ici une expérience limitée, puis expliquer la décision de la conserver ou non.

## Questions de sortie

- Pourquoi faut-il masquer les tokens de padding ?
- Quelle différence entre un encodeur gelé et une tête de classification gelée ?

**Remise :** un run traçable, une conclusion, un artefact ou une soumission valide et une prochaine hypothèse. Aucun score public ni prix garanti.